<a href="https://colab.research.google.com/github/alter-mix-dev/wasapcloudapi/blob/feat%2Fpedidos/AGENTE_PEDIDOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# BLOCK 1: PREPARACIÓN DEL ENTORNO Y CREDENCIALES
#En este bloque instalamos todas las librerías necesarias de Python y del sistema,
# importamos los módulos globales y cargamos de forma segura mis credenciales desde Colab Secrets

In [2]:
# 1.1 INSTALACIÓN DE DEPENDENCIAS
# Instalamos componentes del sistema y librerías de Python requeridas en un solo paso
!apt-get update -qq
!apt-get install -y zstd -qq  # Requerido por Ollama
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama fastapi uvicorn pyngrok requests pytz --quiet

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
# 1.2 IMPORTS GLOBALES Y CONFIGURACIÓN HORARIA
# ==============================================================================
import os
import time
import threading
import subprocess
import requests
from datetime import datetime
import pytz

import ollama
import uvicorn
from fastapi import FastAPI, Request, BackgroundTasks
from pyngrok import ngrok
from google.colab import userdata


In [4]:
# 1.3 CARGA DE CREDENCIALES (COLAB SECRETS)
# ==============================================================================
VERIFY_TOKEN = "mi_token"  # Debe coincidir con el configurado en Meta for Developers
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')  # mi nuevo token de 2 meses
PHONE_ID = userdata.get('PHONE_NUMBER_ID') # Phone Number ID
WABA_ID = "1061287516789190"  # Identificador de la cuenta de WhatsApp Business

print("✅ Entorno preparado e imports cargados.")
print("✅ Credenciales leídas desde Secrets correctamente.")

✅ Entorno preparado e imports cargados.
✅ Credenciales leídas desde Secrets correctamente.


In [5]:
#BLOCK 2: BACKEND DE IA (OLLAMA & MODELO)
#Este bloque se encarga de inicializar el servidor local de Ollama en la máquina virtual de Colab en segundo plano
# y descargar el modelo Llama 3.1.

In [6]:
# 2.1 ARRANQUE DEL SERVIDOR OLLAMA (tarda como 2 minutos)
proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # Tiempo de espera para levantar el servicio local

# Descarga del modelo específico
!ollama pull llama3.1:8b
print("✅ Servidor de Ollama y modelo Llama 3.1 listos.")


✅ Servidor de Ollama y modelo Llama 3.1 listos.


In [7]:
# 2.2 INSTALAR Y LEVANTAR LLAMA STACK
!pip install "llama-stack[starter]" llama-stack-client --quiet
!pip install "mcp<2.0" --quiet

In [8]:
# 2.3. Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado
import subprocess
import time
import os
import yaml
import llama_stack

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

# Generar un config de Llama Stack con el modelo de Ollama ya registrado
config_path = os.path.join(os.path.dirname(llama_stack.__file__), "distributions", "starter", "config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:8b",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:8b",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Levantar el servidor con ese config
proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321")
    print("Llama Stack listo.")

Proceso vivo: True
Llama Stack listo.


In [9]:
#BLOCK 3: LÓGICA DE NEGOCIO Y MODELO DE CONTEXTO
#Definimos la lógica del asistente de comida. El modelo recibe inyección dinámica del día y la hora de México
# para responder de manera coherente si el usuario pregunta por horarios o membresía o estatus de pedidos.

In [10]:
# 3.1 BASE DE DATOS EN MEMORIA
# ==============================================================================
mensajes_recibidos = []
usuarios_conocidos = set()  # Almacena los números que ya recibieron el saludo inicial
contador_pedidos = 1  # Lleva el consecutivo global
historial_pedidos = {}  # Relaciona { numero_telefono: [lista de pedidos] }
historial_conversacion = {} # Guarda el contexto de la charla { numero_telefono: [lista_de_mensajes_ia] }

In [11]:
# 3.2 HELPERS Y GENERACIÓN DE RESPUESTA CON MÁQUINA DE ESTADOS
# ==============================================================================
def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

def generar_respuesta(texto_usuario, numero_usuario):
    global contador_pedidos

    # Variables de control temporal
    zona_horaria = pytz.timezone('America/Mexico_City')
    ahora = datetime.now(zona_horaria)
    fecha_actual = ahora.strftime("%A, %d de %B de %Y")
    hora_actual = ahora.strftime("%H:%M")

    cliente_ollama = ollama.Client(host="http://127.0.0.1:11434")
    folio_actual = contador_pedidos

    texto_clean = texto_usuario.strip().lower()

    # --------------------------------------------------------------------------
    # FLUJO EXCLUSIVO: CONSULTA DE ESTATUS (PERSISTENTE)
    # --------------------------------------------------------------------------
    if "estatus" in texto_clean or "estado de mi pedido" in texto_clean:
        if numero_usuario in historial_pedidos and len(historial_pedidos[numero_usuario]) > 0:
            ultimo_pedido = historial_pedidos[numero_usuario][-1]
            metodo = ultimo_pedido.get("metodo_entrega", "No especificado")
            return (f"📋 El estatus de tu pedido #{ultimo_pedido['folio']} es: *En preparación* 🍳.\n"
                    f"📍 Tipo de entrega: *{metodo}*.\n"
                    f"⏰ Registrado a las: {ultimo_pedido['fecha_hora_texto']}.")
        else:
            return "❌ No encontré ningún pedido activo asociado a tu número de teléfono. ¿Qué deseas ordenar hoy? 🍔"

    # Inicializar el historial de pedidos si no existe
    if numero_usuario not in historial_pedidos:
        historial_pedidos[numero_usuario] = []

    # --------------------------------------------------------------------------
    # MÁQUINA DE ESTADOS: PROCESAMIENTO DE ENTREGA Y CIERRE
    # --------------------------------------------------------------------------
    # Verificamos si el usuario tiene un proceso de cierre activo en curso
    pedido_pendiente_cierre = next((p for p in historial_pedidos[numero_usuario] if not p.get("finalizado", False)), None)

    if pedido_pendiente_cierre:
        folio_pendiente = pedido_pendiente_cierre["folio"]

        # Estado A: Esperando definición del método de entrega
        if pedido_pendiente_cierre.get("sub_estado") == "ESPERANDO_METODO":
            if "domicilio" in texto_clean or "repartidor" in texto_clean or "envio" in texto_clean:
                pedido_pendiente_cierre["metodo_entrega"] = "A domicilio"
                pedido_pendiente_cierre["sub_estado"] = "ESPERANDO_DIRECCION"
                return "📍 ¡Perfecto! Por favor, compárteme tu ubicación actual por WhatsApp o escribe tu dirección completa con referencias."

            elif "pasare" in texto_clean or "recoger" in texto_clean or "sucursal" in texto_clean or "tienda" in texto_clean or "adios" in texto_clean:
                pedido_pendiente_cierre["metodo_entrega"] = "Recoger en sucursal"
                pedido_pendiente_cierre["finalizado"] = True
                contador_pedidos += 1  # Incrementamos el folio global de manera segura hasta este punto

                # Limpiamos el historial de chat con la IA de este pedido anterior
                if numero_usuario in historial_conversacion:
                    del historial_conversacion[numero_usuario]

                return (f"✅ ¡Entendido! Tu pedido #{folio_pendiente} ha sido registrado para *Recoger en sucursal*.\n"
                        f"🏪 Estará listo en aproximadamente 20-25 minutos. ¡Te esperamos! 👋")
            else:
                return "🤔 No logré comprender tu elección. ¿Prefieres que te lo llevemos *a domicilio* o pasas a *recoger en sucursal*?"

        # Estado B: Esperando la dirección de envío
        if pedido_pendiente_cierre.get("sub_estado") == "ESPERANDO_DIRECCION":
            pedido_pendiente_cierre["direccion"] = texto_usuario
            pedido_pendiente_cierre["finalizado"] = True
            contador_pedidos += 1

            if numero_usuario in historial_conversacion:
                del historial_conversacion[numero_usuario]

            return (f"🚀 ¡Todo listo! Tu pedido #{folio_pendiente} ha sido confirmado para envío a domicilio.\n"
                    f"🏠 Dirección de entrega: {texto_usuario}.\n"
                    f"⏱️ Tu orden llegará en aproximadamente 35-45 minutos.")
            # --------------------------------------------------------------------------
    # PROMPT ASISTENTE DE PEDIDOS: TOMA DE PEDIDO CON IA (OLLAMA)
    # --------------------------------------------------------------------------
    if numero_usuario not in historial_conversacion:
        prompt_sistema = (
            f"Eres un agente de toma de pedidos automatizado para nuestro restaurante.\n"
            f"Tu objetivo es ser amable, eficiente y limitarte ESTRICTAMENTE a vender lo que está en el menú.\n\n"
            f"--- MENÚ DISPONIBLE Y PRECIOS ---\n"
            f"• Hamburguesa Clásica con papas: $120 MXN\n"
            f"• Hamburguesa con Queso y tocino: $145 MXN\n"
            f"• Papas a la francesa individuales: $45 MXN\n"
            f"• Refrescos (Cola, Manzana, Limón): $35 MXN\n"
            f"• Agua del día (Litro): $40 MXN\n"
            f"----------------------------------\n\n"
            f"CONTEXTO DE OPERACIÓN:\n"
            f"- Teléfono del cliente: {numero_usuario}\n"
            f"- Siguiente Folio de Pedido disponible: #{folio_actual}\n\n"
            f"REGLAS CRÍTICAS:\n"
            f"1. Recuerda los productos que el usuario ya agregó. No los vuelvas a preguntar.\n"
            f"2. Cuando el usuario te pida explícitamente finalizar, cerrar o terminar la orden, imprime obligatoriamente el resumen detallado, menciona expresamente el texto 'Total: $X MXN' y pregunta textualmente de forma exacta: '¿Deseas tu pedido a domicilio o prefieres recoger en sucursal?'\n"
            f"3. Da respuestas concisas de máximo 3 líneas.\n"
            f"4. NO limpies ni despidas la sesión tú mismo usando palabras como adiós."
        )
        historial_conversacion[numero_usuario] = [{"role": "system", "content": prompt_sistema}]

    historial_conversacion[numero_usuario].append({"role": "user", "content": texto_usuario})

    respuesta = cliente_ollama.chat(model="llama3.1:8b", messages=historial_conversacion[numero_usuario])
    contenido_respuesta = respuesta["message"]["content"]

    historial_conversacion[numero_usuario].append({"role": "assistant", "content": contenido_respuesta})

    # INTERCEPCIÓN DE CIERRE: Si la IA detecta que el usuario quiere finalizar y ofrece las opciones de entrega
    if "recoger en sucursal" in contenido_respuesta.lower() or "domicilio" in contenido_respuesta.lower():
        # Pre-guardamos el pedido sin marcarlo como "finalizado" para congelar sus datos y el folio
        historial_pedidos[numero_usuario].append({
            "folio": folio_actual,
            "detalle_texto": texto_usuario,
            "objeto_tiempo": ahora,
            "fecha_hora_texto": ahora.strftime("%d/%m/%Y %H:%M"),
            "finalizado": False,
            "sub_estado": "ESPERANDO_METODO"
        })
        print(f"⏳ PEDIDO EN PROCESO DE CIERRE -> Reservando Folio: #{folio_actual}")

    return contenido_respuesta

In [12]:
#BLOCK 4: DEFINICIÓN DEL SERVIDOR WEBHOOK (FASTAPI)
#Configuramos nuestra aplicación FastAPI con los endpoints requeridos por Meta. Procesamos los mensajes
#entrantes en segundo plano (BackgroundTasks) para mandar una confirmación HTTP inmediata a WhatsApp y evitar duplicidades.

In [13]:
#_ = '''
# 4.1 ENDPOINTS DE FASTAPI
# ==============================================================================
app = FastAPI()

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params
    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))
    return {"error": "token invalido"}

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass

    return {"status": "ok"}  # Meta recibe el ACK inmediato


In [14]:
#_ = '''
# 4.2 ACCIONES DEL WEBHOOK
# ==============================================================================
def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v25.0/{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }
    requests.post(url, headers=headers, json=payload)

def procesar_y_responder(numero, texto):
    if numero not in usuarios_conocidos:
        usuarios_conocidos.add(numero)
        enviar_respuesta(numero, "¡Hola! Soy el asistente encargado de tomar tu pedido. ¿Qué te gustaría ordenar?")
    else:
        print(f"🤖 Consultando a Ollama para el cliente {numero}...")

        # Ojo aquí: Estas líneas ahora tienen 8 espacios al inicio para estar DENTRO del else
        respuesta_ia = generar_respuesta(texto, numero)
        print(f"✍️ Respuesta generada por la IA: '{respuesta_ia}'")
        enviar_respuesta(numero, respuesta_ia)

# Esta línea queda al ras porque es un aviso general de tu consola
print("✅ Servidor Webhook declarado.")

✅ Servidor Webhook declarado.


In [15]:
#BLOCK 5: DESPLIEGUE, TÚNEL NGROK Y PRUEBAS
#Este bloque expone tu servidor local al internet público por medio de un túnel ngrok y lanza las simulaciones de prueba.
#Incluye el código muerto de tus diagnósticos y la prueba explícita para tu nuevo token.

In [16]:
# 5.1 ACTIVACIÓN DE TÚNEL Y SERVIDOR
# ==============================================================================
ngrok.kill()  # Limpiamos cualquier túnel previo trabado
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
tunel_publico = ngrok.connect(8000)

print("🔗 URL a configurar como Callback URL en el dashboard de Meta:")
print(tunel_publico.public_url + "/webhook\n")

def correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000)

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()
time.sleep(2)  # Dar tiempo para asegurar el arranque


🔗 URL a configurar como Callback URL en el dashboard de Meta:
https://mutate-stylus-facedown.ngrok-free.dev/webhook



INFO:     Started server process [62167]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [17]:
# 5.2 SIMULACIÓN DE FLUJO COMPLETO (LOCAL)
# ==============================================================================
payload_1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "5215614857733", "text": {"body": "Hola"}}
    ]}}]}]
}
r1 = requests.post("http://127.0.0.1:8000/webhook", json=payload_1)
print("🧪 Simulación de primer mensaje (Saludo):", r1.status_code, r1.text)


INFO:     127.0.0.1:43346 - "POST /webhook HTTP/1.1" 200 OK
🧪 Simulación de primer mensaje (Saludo): 200 {"status":"ok"}


In [18]:
_ = '''
# ==============================================================================
# CELDA AISLADA DE PRUEBA: MÚLTIPLES USUARIOS SIMULTÁNEOs
# ==============================================================================
import json
import time
import requests

print("🚀 --- INICIANDO PRUEBA DE CONCURRENCIA DE USUARIOS --- \n")

# Definimos dos números de teléfono distintos
CLIENTE_A = "5215500000001"
CLIENTE_B = "5215500000002"

# 1. El Cliente A inicia pidiendo algo costoso
payload_a1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": CLIENTE_A, "text": {"body": "Hola, quiero una Hamburguesa con Queso y tocino"}}
    ]}}]}]
}

# 2. El Cliente B (en paralelo) inicia pidiendo un complemento barato
payload_b1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": CLIENTE_B, "text": {"body": "Buenas, me das unas Papas a la francesa individuales"}}
    ]}}]}]
}

# 3. El Cliente A decide pedir su cuenta
payload_a2 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": CLIENTE_A, "text": {"body": "Sería todo, la cuenta por favor"}}
    ]}}]}]
}

# --- EJECUCIÓN DE LA SIMULACIÓN ---

print(f"👤 [CLIENTE A - {CLIENTE_A}]: Enviando pedido...")
r_a1 = requests.post("http://127.0.0.1:8000/webhook", json=payload_a1)
time.sleep(3)  # Damos 3 segundos completos para que Ollama procese

print(f"\n👤 [CLIENTE B - {CLIENTE_B}]: Enviando pedido...")
r_b1 = requests.post("http://127.0.0.1:8000/webhook", json=payload_b1)
time.sleep(3)

print(f"\n👤 [CLIENTE A - {CLIENTE_A}]: Solicitando cierre de cuenta...")
r_a2 = requests.post("http://127.0.0.1:8000/webhook", json=payload_a2)
time.sleep(3)

print("\n📊 --- VERIFICACIÓN DE CONTEXTOS ---")
print(f"Estado de la petición final: {r_a2.status_code} OK")
'''

In [19]:
_ = '''
 #CELDA AISLADA: SIMULAR USUARIO FANTASMA PARA INTERCALAR CON MI WHATSAPP REAL
# ==============================================================================
import time

CLIENTE_FANTASMA = "5215599999999"
MI_NUMERO_REAL = "525614857733"

print("🔥 --- INICIANDO SIMULACIÓN DE INTERFERENCIA ---")
print("El cliente fantasma agregará un producto al sistema en este momento...\n")

# El usuario falso pide unas papas (costo $45 MXN)
texto_fantasma = "Buenas, me das unas Papas a la francesa individuales y sería todo"
procesar_y_responder(CLIENTE_FANTASMA, texto_fantasma)

print("✅ [SISTEMA]: Papas agregadas al carrito del Cliente Fantasma.")
print(f"👉 AHORA VE A TU WHATSAPP (Número: {MI_NUMERO_REAL}) Y HAZ TU PEDIDO.")
print("Escribe algo diferente (ej: 'Hola, quiero una Hamburguesa Clásica y la cuenta').")
print("Verás que el bot te cobrará solo tus $120 MXN, ignorando el pedido del fantasma.")
'''

In [20]:
_= '''
# 5.3 PRUEBA EXCLUSIVA PARA EL TOKEN DE 2 MESES (CONEXIÓN CON META)
# ==============================================================================
print("\n🔍 --- PROBANDO VALIDEZ DE TU NUEVO TOKEN DE 2 MESES ---")
# Intentamos vincular la app con el WABA ID usando el nuevo token de secrets
r_token = requests.post(
    f"https://graph.facebook.com/{1061287516789190}/subscribed_apps",
    headers={"Authorization": f"Bearer {WHATSAPP_TOKEN}"}
)

if r_token.status_code == 200:
    print("✅ ¡ÉXITO! Meta aceptó el token correctamente. Código 200 OK.")
    print("Detalle:", r_token.text)
else:
    print(f"❌ ERROR {r_token.status_code}: El token guardado en Secrets sigue dando problemas.")
    print("Respuesta de Meta:", r_token.text)
    '''

In [21]:
# 5.4 CÓDIGO MUERTO DE PRUEBAS ADICIONALES Y DIAGNÓSTICOS ANTERIORES
# ==============================================================================
_ = '''
payload_2 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "5215614857733", "text": {"body": "¿A qué hora abren los sábados?"}}
    ]}}]}]
}
r2 = requests.post("http://127.0.0", json=payload_2)
print("Segundo mensaje:", r2.status_code, r2.text)
'''

_ = '''
if 'mensajes_recibidos' not in locals(): mensajes_recibidos = []
if 'respuestas_enviadas' not in locals(): respuestas_enviadas = 0
if 'errores_procesamiento' not in locals(): errores_procesamiento = 0
if 'conversaciones_por_numero' not in locals(): conversaciones_por_numero = {}

print("--- DIAGNÓSTICO DEL BOT ---")
print("Mensajes recibidos:", len(mensajes_recibidos))
print("Respuestas enviadas:", respuestas_enviadas)
print("Errores:", errores_procesamiento)
try: print("Llama Stack vivo:", proceso_llama_stack.poll() is None)
except NameError: print("Llama Stack vivo: Error (Variable no definida)")
try: print("Servidor vivo:", hilo.is_alive())
except NameError: print("Servidor vivo: Error (Variable no definida)")
'''

_ = '''
url = tunel_publico.public_url + "/webhook"
r = requests.get(url, params={"hub.verify_token": VERIFY_TOKEN, "hub.challenge": "12345"})
print("Prueba Túnel:", r.status_code, r.text)
'''

In [22]:
_ = '''
import socket

def verificar_puerto(puerto, nombre_servicio):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(2)
    resultado = sock.connect_ex(('127.0.0.1', puerto))
    if resultado == 0:
        print(f"🟢 PUERTO {puerto} [{nombre_servicio}]: ACTIVADO y escuchando correctamente.")
    else:
        print(f"🔴 PUERTO {puerto} [{nombre_servicio}]: CERRADO o inactivo. (Código de error: {resultado})")
    sock.close()

print("--- AUDITORÍA DE RED EN TIEMPO REAL ---")
verificar_puerto(11434, "Servidor Interno de Ollama")
verificar_puerto(8000, "Tu Servidor Webhook FastAPI")
'''